In [2]:
# imports
import pandas as pd
import numpy as np

In [3]:
# loading in the data

columns = [
    'variance',
    'skewness',
    'curtosis',
    'entropy',
    'label'
]

train_data = pd.read_csv('bank-note/train.csv', names=columns,header=None)
test_data = pd.read_csv('bank-note/test.csv', names=columns, header=None)

train_data

,variance,skewness,curtosis,entropy,label
0,3.848100,10.15390,-3.85610,-4.22280,0
1,4.004700,0.45937,1.36210,1.61810,0
2,-0.048008,-1.60370,8.47560,0.75558,0
3,-1.266700,2.81830,-2.42600,-1.88620,1
4,2.203400,5.99470,0.53009,0.84998,0
...,...,...,...,...,...
867,0.273310,4.87730,-4.91940,-5.81980,1
868,1.063700,3.69570,-4.15940,-1.93790,1
869,-1.242400,-1.71750,-0.52553,-0.21036,1
870,1.837300,6.12920,0.84027,0.55257,0


In [4]:
# need to convert the labels from [0,1] -> [-1,1]
# need to change the labels from [-1,1]
train_data.iloc[:, -1].replace(0, -1, inplace=True)
test_data.iloc[:, -1].replace(0, -1, inplace=True)

train_data

/var/folders/h0/f9d5nsss0s14zm26734cgfg80000gn/T/ipykernel_78720/2661093999.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_data.iloc[:, -1].replace(0, -1, inplace=True)
/var/folders/h0/f9d5nsss0s14zm26734cgfg80000gn/T/ipykernel_78720/2661093999.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values a

,variance,skewness,curtosis,entropy,label
0,3.848100,10.15390,-3.85610,-4.22280,-1
1,4.004700,0.45937,1.36210,1.61810,-1
2,-0.048008,-1.60370,8.47560,0.75558,-1
3,-1.266700,2.81830,-2.42600,-1.88620,1
4,2.203400,5.99470,0.53009,0.84998,-1
...,...,...,...,...,...
867,0.273310,4.87730,-4.91940,-5.81980,1
868,1.063700,3.69570,-4.15940,-1.93790,1
869,-1.242400,-1.71750,-0.52553,-0.21036,1
870,1.837300,6.12920,0.84027,0.55257,-1


In [6]:
# now need to split the data to features (columns) and labels (prediction)

x_train = train_data.drop('label', axis=1).values
y_train = train_data['label'].values

x_test = test_data.drop('label', axis=1).values
y_test = test_data['label'].values

# 2: SVM Primal Domain

In [7]:
#initializing variables
# max epochs 100
epoch = 100
#shuffle at the start of each epoch
c = [(100/873), (500/873), (700/873)]
#making a zero weight vector with the number of features
weights = np.zeros(x_train.shape[1])

bias = 0.0
N = len(x_train)
learning_rate = 0.01 #((rate)/(1+((rate/a)*t)))
a = 0.01

In [8]:
# the gradient is the hindge loss

def hinge_loss(w, b, x_i, y_i, c):
    loss_max = np.maximum(0, 1 - (y_i * (np.dot(x_i, w) + b)))
    reg_term = 0.5 * np.dot(w,w)
    return np

In [9]:
#implement SVM in primal domain with stochastic sub-gradient descent

def Primal_SVM_1(x_train, y_train, bias, N, epoch, weights, C, learning_rate, a):
    for t in range(epoch):
        # shuffle at the start of each epoch
        shuffle_idx = np.random.permutation(len(x_train))
        x_train_shuffled = x_train[shuffle_idx]
        y_train_shuffled = y_train[shuffle_idx]

        for feature, label in zip(x_train_shuffled, y_train_shuffled):
            #might need to add bias to wieghts and augment 1 in data for bias
            learning_rate = (learning_rate/(1+ (learning_rate/a)*t))
            predict = label * (np.dot(weights, feature) + bias)

            if(predict < 1):
                #miss classiffied example
                #might need to fix the first parenthesis
                weights = weights - (learning_rate * (-label*feature)) + (learning_rate * C * N * label * feature)
                bias = bias - (learning_rate * -label)
            else:
                # gradient is 0
            
                weights = (1 - learning_rate)*weights

    return weights, bias

In [10]:
# now primal with other learning rate

def Primal_SVM_2(x_train, y_train, bias, N, epoch, weights, C, learning_rate):
    for t in range(epoch):
        # shuffle at the start of each epoch
        shuffle_idx = np.random.permutation(len(x_train))
        x_train_shuffled = x_train[shuffle_idx]
        y_train_shuffled = y_train[shuffle_idx]

        for feature, label in zip(x_train_shuffled, y_train_shuffled):
            #might need to add bias to wieghts and augment 1 in data for bias
            learning_rate = (learning_rate/(1+t))
            predict = label * (np.dot(weights, feature) + bias)

            if(predict < 1):
                #miss classiffied example
                #might need to fix the first parenthesis
                weights = weights - (learning_rate * (-label*feature)) + (learning_rate * C * N * label * feature)
                bias = bias - (learning_rate * -label)
            else:
                # gradient is 0

                weights = (1 - learning_rate)*weights

    return weights, bias

In [11]:
def Error_val(feature, labels, weights, bias):
    predict = np.sign(np.dot(feature, weights) + bias)
    return np.mean(predict != labels)

In [12]:
# finding error
SVM_first_learning_rate = []
SVM_first_LR = []
for C in c: 
    w_first, b_first = Primal_SVM_1(x_train, y_train, bias, N, epoch, weights, C, learning_rate, a)
    
    train_err_first = Error_val(x_train, y_train, w_first, b_first)
    
    test_err_first = Error_val(x_test, y_test, w_first, b_first)
    
    SVM_first_learning_rate.append((w_first, b_first, train_err_first, test_err_first))
    SVM_first_LR.append((C, train_err_first, test_err_first))

In [13]:
print(SVM_first_learning_rate)

[(array([-9.649851  , -5.56005128, -2.8713111 , -4.92138687]), np.float64(0.6386117694656254), np.float64(0.06880733944954129), np.float64(0.082)), (array([-61.24888332, -45.90636385, -49.50107321,  -9.87287973]), np.float64(0.4634896574046418), np.float64(0.05733944954128441), np.float64(0.062)), (array([-54.81846617, -38.28040893, -41.59182096,  -4.17258876]), np.float64(0.6159774930023713), np.float64(0.06192660550458716), np.float64(0.07))]


In [14]:
print(SVM_first_LR)

[(0.1145475372279496, np.float64(0.06880733944954129), np.float64(0.082)), (0.572737686139748, np.float64(0.05733944954128441), np.float64(0.062)), (0.8018327605956472, np.float64(0.06192660550458716), np.float64(0.07))]


In [15]:
SVM_second_learning_rate = []
SVM_second_LR = []
for C in c:
    w_second, b_second = Primal_SVM_2(x_train, y_train, bias, N, epoch, weights, C, learning_rate)

    train_err_second = Error_val(x_train, y_train, w_second, b_second)

    test_err_second = Error_val(x_test, y_test, w_second, b_second)

    SVM_second_learning_rate.append((w_second, b_second, train_err_second, test_err_second))
    SVM_second_LR.append((C, train_err_second, test_err_second))

In [16]:
print(SVM_second_learning_rate)

[(array([-14.40015223, -15.80376092,  -2.40265323,  -5.91582129]), np.float64(0.6200783598480806), np.float64(0.2190366972477064), np.float64(0.196)), (array([-41.81090775, -48.64223791, -62.52776371,   2.34167859]), np.float64(0.5301977634429935), np.float64(0.19839449541284404), np.float64(0.22)), (array([-84.07240086, -49.84443588, -44.66687765, -28.19487279]), np.float64(0.6700097656622896), np.float64(0.0481651376146789), np.float64(0.056))]


In [17]:
print(SVM_second_LR)

[(0.1145475372279496, np.float64(0.2190366972477064), np.float64(0.196)), (0.572737686139748, np.float64(0.19839449541284404), np.float64(0.22)), (0.8018327605956472, np.float64(0.0481651376146789), np.float64(0.056))]


In [18]:
# now compare the two
comparison_results = []
for i, C in enumerate(c):
    w_diff = np.linalg.norm(SVM_first_learning_rate[i][0] - SVM_second_learning_rate[i][0])
    b_diff = abs(SVM_first_learning_rate[i][1] - SVM_second_learning_rate[i][1])
    train_error_diff = abs(SVM_first_learning_rate[i][2] - SVM_second_learning_rate[i][2])
    test_error_diff = abs(SVM_first_learning_rate[i][3] - SVM_second_learning_rate[i][3])
    comparison_results.append({
        'C': C,
        'Diff in w': w_diff,
        'Diff in b': b_diff,
        'Diff in Train Error': train_error_diff,
        'Diff in Test Error': test_error_diff
    })

In [19]:
print("First Schedule Results:", SVM_first_learning_rate)
print("Alternate Schedule Results:", SVM_second_learning_rate)
print("Comparison Results:", comparison_results)

First Schedule Results: [(array([-9.649851  , -5.56005128, -2.8713111 , -4.92138687]), np.float64(0.6386117694656254), np.float64(0.06880733944954129), np.float64(0.082)), (array([-61.24888332, -45.90636385, -49.50107321,  -9.87287973]), np.float64(0.4634896574046418), np.float64(0.05733944954128441), np.float64(0.062)), (array([-54.81846617, -38.28040893, -41.59182096,  -4.17258876]), np.float64(0.6159774930023713), np.float64(0.06192660550458716), np.float64(0.07))]
Alternate Schedule Results: [(array([-14.40015223, -15.80376092,  -2.40265323,  -5.91582129]), np.float64(0.6200783598480806), np.float64(0.2190366972477064), np.float64(0.196)), (array([-41.81090775, -48.64223791, -62.52776371,   2.34167859]), np.float64(0.5301977634429935), np.float64(0.19839449541284404), np.float64(0.22)), (array([-84.07240086, -49.84443588, -44.66687765, -28.19487279]), np.float64(0.6700097656622896), np.float64(0.0481651376146789), np.float64(0.056))]
Comparison Results: [{'C': 0.1145475372279496, '

In [20]:
print(comparison_results)

[{'C': 0.1145475372279496, 'Diff in w': np.float64(11.34493230176625), 'Diff in b': np.float64(0.018533409617544794), 'Diff in Train Error': np.float64(0.15022935779816513), 'Diff in Test Error': np.float64(0.114)}, {'C': 0.572737686139748, 'Diff in w': np.float64(26.5369553859156), 'Diff in b': np.float64(0.0667081060383517), 'Diff in Train Error': np.float64(0.14105504587155965), 'Diff in Test Error': np.float64(0.158)}, {'C': 0.8018327605956472, 'Diff in w': np.float64(39.699439768502565), 'Diff in b': np.float64(0.05403227265991839), 'Diff in Train Error': np.float64(0.013761467889908258), 'Diff in Test Error': np.float64(0.014000000000000005)}]


# SVM in Duel Domain

In [23]:
from scipy.optimize import minimize

In [24]:
def Duel_svm(alpha, features, labels):

    feature_matrix = np.dot(features, features.T)  # Precompute the  matrix
    return 0.5 * np.sum(alpha[:, None] * alpha[None, :] * labels[:, None] * labels[None, :] * feature_matrix) - np.sum(alpha)

In [25]:
c = [(100/873), (500/873), (700/873)]

In [26]:
def constraint(alphas,y):
    return np.dot(alphas, y)

In [27]:
# need to use teh KKT conditions
def Duel_svn_recover(x, y, alphas):

    # if alphas are 0 then will not affect the weight vector

    weight_star = np.sum(alphas[:, None] * y[:, None] * x, axis=0)
    # makesure to det a thres for the non-zero ones
    support_vec = np.where(alphas > 1e-5)[0]


    if len(support_vec) == 0:
        return weight_star, np.nan


    bias_star = np.mean(y[support_vec] - np.dot(x[support_vec], weight_star))

    return weight_star, bias_star

In [28]:
svm_Duel = []
for C in c:
    constraints = ({'type': 'eq', 'fun': constraint, 'args': (y_train,)})
    inital_alpha = np.zeros(len(y_train))
    bounds = [(0, C) for _ in range(len(y_train))]
    result = minimize(Duel_svm, inital_alpha, args=(x_train, y_train),
                      method='SLSQP', bounds=bounds, constraints=constraints)

    optimal_alphas = result.x
    w, b = Duel_svn_recover(x_train, y_train, optimal_alphas)
    #print(w,b)
    svm_Duel.append((w,b,C))

/Users/tailangcao/Desktop/CS5350/HW4/myenv/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:434: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)


In [29]:
print(svm_Duel)

[(array([-0.94292622, -0.65149179, -0.73372191, -0.04102192]), np.float64(1.074503284657922), 0.1145475372279496), (array([-1.56393891, -1.01405258, -1.18065124, -0.15651753]), np.float64(1.5123653981081924), 0.572737686139748), (array([-2.04255405, -1.28070205, -1.51352506, -0.24906694]), np.float64(1.8793654572079423), 0.8018327605956472)]


# Gaussian Kernel

In [30]:
def gaussian_Kernel(x1,x2, gam):

    return np.exp(((np.linalg.norm(x1 - x2)**2)/ gam))

In [31]:
gammas = [0.1, 0.5, 1, 5, 100]
c = [(100/873), (500/873), (700/873)]

In [32]:
def duel_kernel(alpha, features, labels, gamma):
    #use the kernel
    kernal = gaussian_Kernel(features, features, gamma)

    return 0.5 * np.sum(alpha[:, None] * alpha[None, :] * labels[:, None] * labels[None, :] * kernal) - np.sum(alpha)

In [33]:
def predict_with_gauss_train(train_x, y, test_x, alpha, bias, gamma):
    kernal = gaussian_Kernel(train_x, train_x, gamma)
    predict = np.sign(np.sum(kernal * alpha * y) + bias)
    return predict

In [34]:
def predict_with_gauss_test(train_x, y, test_x, alpha, bias, gamma):
 
    
    #to account for different test and train shapes
    num_test_samples = len(test_x)
    num_train_samples = len(train_x)

    predictions = np.zeros(num_test_samples)

    for i in range(num_test_samples):
        kernal = gaussian_Kernel(test_x[i], train_x, gamma)
        predictions[i] = np.sign(np.sum(kernal * alpha[:num_train_samples] * y[:num_train_samples, None]) + bias)

    return predictions

In [35]:
result_kern = {}
support_result = {}
support_result_num={}
for gam in gammas:
    for C in c:
        constraints = ({'type': 'eq', 'fun': constraint, 'args': (y_train,)})
        bounds = [(0, C) for _ in range(len(y_train))]
        initial_alpha = np.zeros(len(y_train))
        result = minimize(duel_kernel, initial_alpha, args=(x_train, y_train, gam),
                          method='SLSQP', bounds=bounds, constraints=constraints)
        
        opt_alphas = result.x
      
        weights, bias = Duel_svn_recover(x_train, y_train, opt_alphas)
        train_predict = predict_with_gauss_train(x_train, y_train, x_train, opt_alphas, bias, gam)
        
        test_predict = predict_with_gauss_test(x_train, y_test, x_test, opt_alphas, bias, gam)
        
        training_err = np.mean(train_predict != y_train)
        test_err = np.mean(test_predict != y_test)
        
        result_kern[(gam, C)] = (training_err, test_err)
        
        support_result[(gam, C)] = opt_alphas
        support_result_num[(gam, C)] = len(opt_alphas)

/var/folders/h0/f9d5nsss0s14zm26734cgfg80000gn/T/ipykernel_78720/1090433918.py:3: RuntimeWarning: overflow encountered in exp
  return np.exp(((np.linalg.norm(x1 - x2)**2)/ gam))
/Users/tailangcao/Desktop/CS5350/HW4/myenv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/Users/tailangcao/Desktop/CS5350/HW4/myenv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


In [36]:
print(result_kern)

{(0.1, 0.1145475372279496): (np.float64(0.5538990825688074), np.float64(1.0)), (0.1, 0.572737686139748): (np.float64(0.5538990825688074), np.float64(1.0)), (0.1, 0.8018327605956472): (np.float64(0.5538990825688074), np.float64(1.0)), (0.5, 0.1145475372279496): (np.float64(0.5538990825688074), np.float64(1.0)), (0.5, 0.572737686139748): (np.float64(0.5538990825688074), np.float64(1.0)), (0.5, 0.8018327605956472): (np.float64(0.5538990825688074), np.float64(1.0)), (1, 0.1145475372279496): (np.float64(0.5538990825688074), np.float64(1.0)), (1, 0.572737686139748): (np.float64(0.5538990825688074), np.float64(1.0)), (1, 0.8018327605956472): (np.float64(0.5538990825688074), np.float64(1.0)), (5, 0.1145475372279496): (np.float64(0.5538990825688074), np.float64(1.0)), (5, 0.572737686139748): (np.float64(0.5538990825688074), np.float64(1.0)), (5, 0.8018327605956472): (np.float64(0.5538990825688074), np.float64(1.0)), (100, 0.1145475372279496): (np.float64(0.5538990825688074), np.float64(0.91)), 

In [37]:
print(support_result)

{(0.1, 0.1145475372279496): array([0.09225464, 0.09225464, 0.09225464, 0.11454754, 0.09225464,
       0.09225464, 0.09225464, 0.09225464, 0.09225464, 0.09225464,
       0.11454754, 0.11454754, 0.09225464, 0.11454754, 0.11454754,
       0.11454754, 0.09225464, 0.09225464, 0.11454754, 0.11454754,
       0.11454754, 0.11454754, 0.11454754, 0.09225464, 0.09225464,
       0.11454754, 0.09225464, 0.09225464, 0.09225464, 0.09225464,
       0.09225464, 0.09225464, 0.11454754, 0.11454754, 0.09225464,
       0.09225464, 0.11454754, 0.11454754, 0.09225464, 0.09225464,
       0.11454754, 0.09225464, 0.09225464, 0.11454754, 0.11454754,
       0.11454754, 0.11454754, 0.11454754, 0.09225464, 0.09225464,
       0.09225464, 0.11454754, 0.11454754, 0.11454754, 0.09225464,
       0.09225464, 0.11454754, 0.09225464, 0.11454754, 0.11454754,
       0.11454754, 0.09225464, 0.11454754, 0.09225464, 0.09225464,
       0.09225464, 0.11454754, 0.09225464, 0.09225464, 0.11454754,
       0.09225464, 0.09225464, 0.1

In [38]:
# Output the results
for key, value in result_kern.items():
    print(f"Gamma: {key[0]}, C: {key[1]} => Train Error: {value[0]}, Test Error: {value[1]}")

Gamma: 0.1, C: 0.1145475372279496 => Train Error: 0.5538990825688074, Test Error: 1.0
Gamma: 0.1, C: 0.572737686139748 => Train Error: 0.5538990825688074, Test Error: 1.0
Gamma: 0.1, C: 0.8018327605956472 => Train Error: 0.5538990825688074, Test Error: 1.0
Gamma: 0.5, C: 0.1145475372279496 => Train Error: 0.5538990825688074, Test Error: 1.0
Gamma: 0.5, C: 0.572737686139748 => Train Error: 0.5538990825688074, Test Error: 1.0
Gamma: 0.5, C: 0.8018327605956472 => Train Error: 0.5538990825688074, Test Error: 1.0
Gamma: 1, C: 0.1145475372279496 => Train Error: 0.5538990825688074, Test Error: 1.0
Gamma: 1, C: 0.572737686139748 => Train Error: 0.5538990825688074, Test Error: 1.0
Gamma: 1, C: 0.8018327605956472 => Train Error: 0.5538990825688074, Test Error: 1.0
Gamma: 5, C: 0.1145475372279496 => Train Error: 0.5538990825688074, Test Error: 1.0
Gamma: 5, C: 0.572737686139748 => Train Error: 0.5538990825688074, Test Error: 1.0
Gamma: 5, C: 0.8018327605956472 => Train Error: 0.5538990825688074, 

In [39]:
# Output the results
for key, value in support_result.items():
    print(f"Gamma: {key[0]}, C: {key[1]} =>support Vectors: {value[0].size}")

Gamma: 0.1, C: 0.1145475372279496 =>support Vectors: 1
Gamma: 0.1, C: 0.572737686139748 =>support Vectors: 1
Gamma: 0.1, C: 0.8018327605956472 =>support Vectors: 1
Gamma: 0.5, C: 0.1145475372279496 =>support Vectors: 1
Gamma: 0.5, C: 0.572737686139748 =>support Vectors: 1
Gamma: 0.5, C: 0.8018327605956472 =>support Vectors: 1
Gamma: 1, C: 0.1145475372279496 =>support Vectors: 1
Gamma: 1, C: 0.572737686139748 =>support Vectors: 1
Gamma: 1, C: 0.8018327605956472 =>support Vectors: 1
Gamma: 5, C: 0.1145475372279496 =>support Vectors: 1
Gamma: 5, C: 0.572737686139748 =>support Vectors: 1
Gamma: 5, C: 0.8018327605956472 =>support Vectors: 1
Gamma: 100, C: 0.1145475372279496 =>support Vectors: 1
Gamma: 100, C: 0.572737686139748 =>support Vectors: 1
Gamma: 100, C: 0.8018327605956472 =>support Vectors: 1


In [40]:
# Output the results
for key, value in support_result.items():
    if key[1] == 500/873:
        print(f"Gamma: {key[0]}, C: {key[1]} =>support Vectors: {value[0].size}")

Gamma: 0.1, C: 0.572737686139748 =>support Vectors: 1
Gamma: 0.5, C: 0.572737686139748 =>support Vectors: 1
Gamma: 1, C: 0.572737686139748 =>support Vectors: 1
Gamma: 5, C: 0.572737686139748 =>support Vectors: 1
Gamma: 100, C: 0.572737686139748 =>support Vectors: 1
